## Kite API

### Connect to Kite API

* Get Kite Connect API Subscrption
* Fetch the Kite Connect API KEY & API SECRET
* Use the URL to login & get the REDIRECT URL
* Connect to the Kite session


In [2]:
#!pip install --upgrade kiteconnect
#!pip install pandas

In [40]:
import pandas as pd
from datetime import datetime
import json
import time
from datetime import datetime, timedelta, time
import pandas as pd
import re


In [41]:
API_KEY = "ynacybo7nnsvhxfv"
API_SECRET = "uzmmqg2rlwz9le8nnkvf158prcq5btv9"

In [42]:
import logging
from kiteconnect import KiteConnect

logging.basicConfig(level=logging.DEBUG)

kite = KiteConnect(api_key=API_KEY)

kite.login_url()

# Redirect the user to the login url obtained
# from kite.login_url(), and receive the request_token
# from the registered redirect url after the login flow.
# Once you have the request_token, obtain the access_token
# as follows.

'https://kite.trade/connect/login?api_key=ynacybo7nnsvhxfv&v=3'

In [62]:
# replace with actual redirect URL
REDIRECT_URL = "http://127.0.0.1:8000/?action=login&type=login&status=success&request_token=Srb3pnmk21KqmeI3d33PuVnIB9tWg3rT" 

In [63]:
import requests
import hashlib

# Example code to extract request_token from redirect URL (replace with your actual implementation)
request_token = REDIRECT_URL.split("request_token=")[1].split("&")[0]
print(f"Request Token: {request_token}")

session_url = "https://kite.zerodha.com/session/token"
checksum = hashlib.sha256(f"{API_KEY}{request_token}{API_SECRET}".encode()).hexdigest()
data = {"request_token": request_token, "checksum": checksum}

REQUEST_TOKEN = data['request_token']

try:
    data = kite.generate_session(REQUEST_TOKEN, api_secret=API_SECRET)
    print("Kite session generated successfully.")
except requests.exceptions.RequestException as e:
    print(f"Error generating access token: {e}")

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "POST /session/token HTTP/1.1" 200 None


Request Token: Srb3pnmk21KqmeI3d33PuVnIB9tWg3rT
Kite session generated successfully.


### API Data

* The API will give data about the instruments & the historical data

In [252]:
instrumentList = pd.read_csv("https://api.kite.trade/instruments")

In [253]:
instrumentList

,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
0,215681797,842507,BANKEX25SEPFUT,BANKEX,0,2025-09-25,0.0,0.05,30,FUT,BFO-FUT,BFO
1,220537093,861473,BANKEX25OCTFUT,BANKEX,0,2025-10-30,0.0,0.05,30,FUT,BFO-FUT,BFO
2,215645189,842364,SENSEX25SEPFUT,SENSEX,0,2025-09-25,0.0,0.05,20,FUT,BFO-FUT,BFO
3,220514309,861384,SENSEX25OCTFUT,SENSEX,0,2025-10-30,0.0,0.05,20,FUT,BFO-FUT,BFO
4,215835653,843108,SENSEX5025SEPFUT,SENSEX50,0,2025-09-25,0.0,0.05,75,FUT,BFO-FUT,BFO
...,...,...,...,...,...,...,...,...,...,...,...,...
90576,194418177,759446,12AFIL27-N0,NaN,0,NaN,0.0,0.01,1,EQ,NSE,NSE
90577,194418689,759448,SHREEJISPG,SHREEJI SHIPPING GLOBAL L,0,NaN,0.0,0.05,1,EQ,NSE,NSE
90578,194419969,759453,VIKRAMSOLR,VIKRAM SOLAR,0,NaN,0.0,0.05,1,EQ,NSE,NSE
90579,194421249,759458,GROWWNXT50,GROWWAMC - GROWWNXT50,0,NaN,0.0,0.01,1,EQ,NSE,NSE


In [254]:
instrumentList['segment'].unique()

array(['BFO-FUT', 'BFO-OPT', 'BSE', 'CDS-FUT', 'CDS-OPT', 'INDICES',
       'MCX-FUT', 'MCX-OPT', 'NCO', 'NCO-FUT', 'NCO-OPT', 'NFO-FUT',
       'NFO-OPT', 'NSE'], dtype=object)

In [257]:
nifty_op = instrumentList[(instrumentList['exchange']=='NFO') 
               & (instrumentList['segment']=='NFO-OPT') 
               & (instrumentList['tradingsymbol'].str.contains('NIFTY'))
               & (instrumentList['expiry'].str.contains('2025-08-14'))] 

# Add a new column 'exchange_tradingsymbol_key' to instrumentList
instrumentList['key'] = list(zip(instrumentList['exchange'], instrumentList['tradingsymbol']))

nifty_op.head()

,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange,key
51492,11389954,44492,NIFTY2581424600CE,NIFTY,0,2025-08-14,24600.0,0.05,75,CE,NFO-OPT,NFO,"(NFO, NIFTY2581424600CE)"
51493,11390210,44493,NIFTY2581424600PE,NIFTY,0,2025-08-14,24600.0,0.05,75,PE,NFO-OPT,NFO,"(NFO, NIFTY2581424600PE)"
51494,11389442,44490,NIFTY2581424550CE,NIFTY,0,2025-08-14,24550.0,0.05,75,CE,NFO-OPT,NFO,"(NFO, NIFTY2581424550CE)"
51495,11389698,44491,NIFTY2581424550PE,NIFTY,0,2025-08-14,24550.0,0.05,75,PE,NFO-OPT,NFO,"(NFO, NIFTY2581424550PE)"
51496,11390978,44496,NIFTY2581424650CE,NIFTY,0,2025-08-14,24650.0,0.05,75,CE,NFO-OPT,NFO,"(NFO, NIFTY2581424650CE)"


In [411]:
instrumentList[(instrumentList['segment']=='INDICES')  ]

,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange,key
29292,256265,1001,NIFTY 50,NIFTY 50,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY 50)"
29293,265,1,SENSEX,SENSEX,0,NaN,0.0,0.0,0,EQ,INDICES,BSE,"(BSE, SENSEX)"
29294,256777,1003,NIFTY MIDCAP 100,NIFTY MIDCAP 100,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY MIDCAP 100)"
29295,274441,1072,BANKEX,BSE INDEX BANKEX,0,NaN,0.0,0.0,0,EQ,INDICES,BSE,"(BSE, BANKEX)"
29296,260105,1016,NIFTY BANK,NIFTY BANK,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY BANK)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
29494,413961,1617,NIFTY IND DEFENCE,NIFTY INDIA DEFENCE,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY IND DEFENCE)"
29495,414217,1618,NIFTY IND TOURISM,NIFTY INDIA TOURISM,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY IND TOURISM)"
29496,414473,1619,NIFTY CAPITAL MKT,NIFTY CAPITAL MARKETS,0,NaN,0.0,0.0,0,EQ,INDICES,NSE,"(NSE, NIFTY CAPITAL MKT)"
29497,414729,1620,BSE1000,BSE 1000,0,NaN,0.0,0.0,0,EQ,INDICES,BSE,"(BSE, BSE1000)"


### Fetch data for Options

* Get the start date
* Fetching the open price of NIFTY on the start date
* Based on the open price, get the ATM strike and 20 strikes around ATM
* Generate trading symbols: CE above ATM, PE below ATM
* Fetch the historical data for the strikes

#### NIFTY Options

In [553]:
start = datetime(2025, 8, 13, 9, 15)     # This Friday

In [554]:
hist = kite.historical_data(256265, start, datetime(2025, 8, 13), interval="day") # NIFTY index token

# Convert to DataFrame
df_hist = pd.DataFrame(hist)

# Convert 'date' column to datetime if not already
df_hist['date'] = pd.to_datetime(df_hist['date'])

nifty_open = int(df_hist['open'][0])

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/day?from=2025-08-13+09%3A15%3A00&to=2025-08-13+00%3A00%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None


In [555]:
# 1. Get current spot price
symbol = 'NIFTY 50'
nifty_ltp = kite.quote('NSE:NIFTY 50')['NSE:NIFTY 50']['last_price']

print("---")
print(f"\nCurrent NIFTY LTP: {nifty_ltp}")
print(f"NIFTY Open on {start.date()}: {nifty_open}")

# 2. Calculate ATM and 20 strikes around
atm_strike = int(round(nifty_open / 50) * 50)

print(f"ATM Strike: {atm_strike}")

strike_range = [atm_strike + i*50 for i in range(-10, 11)]  # 20 strikes around ATM

# 3. Construct expiry symbol format for the weekly (Example for 14-Aug-2025)
expiry_dt = datetime(2025, 8, 14)
expiry_str = f"{expiry_dt.year % 100}{expiry_dt.month:d}{expiry_dt.day:02d}"  # '25814'

# 4. Generate trading symbols: CE above ATM, PE below ATM
trading_symbols = []
for strike in strike_range:
    if strike < atm_strike:
        trading_symbols.append(f"NIFTY{expiry_str}{strike}PE")
    elif strike > atm_strike:
        trading_symbols.append(f"NIFTY{expiry_str}{strike}CE")


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=NSE%3ANIFTY+50 HTTP/1.1" 200 None


---

Current NIFTY LTP: 24980.65
NIFTY Open on 2025-08-13: 24586
ATM Strike: 24600


In [556]:
trading_symbols

['NIFTY2581424100PE',
 'NIFTY2581424150PE',
 'NIFTY2581424200PE',
 'NIFTY2581424250PE',
 'NIFTY2581424300PE',
 'NIFTY2581424350PE',
 'NIFTY2581424400PE',
 'NIFTY2581424450PE',
 'NIFTY2581424500PE',
 'NIFTY2581424550PE',
 'NIFTY2581424650CE',
 'NIFTY2581424700CE',
 'NIFTY2581424750CE',
 'NIFTY2581424800CE',
 'NIFTY2581424850CE',
 'NIFTY2581424900CE',
 'NIFTY2581424950CE',
 'NIFTY2581425000CE',
 'NIFTY2581425050CE',
 'NIFTY2581425100CE']

In [338]:
# 5. Load instrument list and map trading symbols to tokens
instrument_list = pd.DataFrame(kite.instruments('NFO'))

nifty_op = instrumentList[(instrumentList['exchange']=='NFO') 
               & (instrumentList['segment']=='NFO-OPT') 
               & (instrumentList['tradingsymbol'].str.contains('NIFTY'))
               & (instrumentList['expiry'].str.contains('2025-08-14')) &
               (instrumentList['tradingsymbol'].isin(trading_symbols))] 

# Add a new column 'exchange_tradingsymbol_key' to instrumentList
nifty_op['key'] = list(zip(nifty_op['exchange'], nifty_op['tradingsymbol']))

# Now build the token_map
token_map = {
    row['tradingsymbol']: row['instrument_token']
    for _, row in nifty_op.iterrows()
}

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/NFO HTTP/1.1" 200 405055
C:\Users\harip\AppData\Local\Temp\ipykernel_35704\2643342579.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nifty_op['key'] = list(zip(nifty_op['exchange'], nifty_op['tradingsymbol']))


In [339]:
token_map 

{'NIFTY2581424600CE': 11389954,
 'NIFTY2581424650CE': 11390978,
 'NIFTY2581424500PE': 11385090,
 'NIFTY2581424700CE': 11391490,
 'NIFTY2581424450PE': 11384578,
 'NIFTY2581424750CE': 11392002,
 'NIFTY2581424400PE': 11384066,
 'NIFTY2581424800CE': 11393026,
 'NIFTY2581424350PE': 11383042,
 'NIFTY2581424850CE': 11393538,
 'NIFTY2581424300PE': 11382530,
 'NIFTY2581424900CE': 11394050,
 'NIFTY2581424250PE': 11382018,
 'NIFTY2581424950CE': 11394562,
 'NIFTY2581425050CE': 11395586,
 'NIFTY2581425000CE': 11395074,
 'NIFTY2581424200PE': 11381506,
 'NIFTY2581424150PE': 11380994,
 'NIFTY2581424100PE': 11380482,
 'NIFTY2581424050PE': 11379458}

In [361]:
start = datetime(2025, 8, 8, 9, 15)     # This Friday
end   = datetime(2025, 8, 14, 15, 30)   # Next Thursday

In [362]:
# Get tokens for only these symbols
tokens_to_fetch = {symbol: token_map.get(symbol) for symbol in trading_symbols}

all_hist_df = pd.DataFrame()

for symbol, token in tokens_to_fetch.items():
    if token is not None:
        try:
            hist = kite.historical_data(token, start, end, interval="5minute")
            df_hist = pd.DataFrame(hist)
            df_hist["symbol"] = symbol
            df_hist["token"] = token
            all_hist_df = pd.concat([all_hist_df, df_hist], ignore_index=True)
            time.sleep(0.5)  # To avoid rate limits
        except Exception as e:
            print(f"Error fetching data for {symbol}: {e}")
    else:
        print(f"Token not found for {symbol}")

display(all_hist_df)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/11379458/5minute?from=2025-08-08+09%3A15%3A00&to=2025-08-14+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/11380482/5minute?from=2025-08-08+09%3A15%3A00&to=2025-08-14+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/11380994/5minute?from=2025-08-08+09%3A15%3A00&to=2025-08-14+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:htt

,date,open,high,low,close,volume,symbol,token
0,2025-08-08 09:15:00+05:30,16.00,16.00,12.55,14.90,893775,NIFTY2581424050PE,11379458
1,2025-08-08 09:20:00+05:30,14.90,16.15,14.20,15.60,685050,NIFTY2581424050PE,11379458
2,2025-08-08 09:25:00+05:30,15.60,20.80,15.05,20.80,948975,NIFTY2581424050PE,11379458
3,2025-08-08 09:30:00+05:30,20.75,22.00,19.15,21.15,706125,NIFTY2581424050PE,11379458
4,2025-08-08 09:35:00+05:30,21.15,23.45,18.10,19.05,873000,NIFTY2581424050PE,11379458
...,...,...,...,...,...,...,...,...
1595,2025-08-08 15:30:00+05:30,6.65,6.65,6.65,6.65,0,NIFTY2581425050CE,11395586
1596,2025-08-08 16:15:00+05:30,6.65,6.65,6.65,6.65,0,NIFTY2581425050CE,11395586
1597,2025-08-08 16:40:00+05:30,6.65,6.65,6.65,6.65,0,NIFTY2581425050CE,11395586
1598,2025-08-08 16:45:00+05:30,6.65,6.65,6.65,6.65,0,NIFTY2581425050CE,11395586


In [359]:
all_hist_df.to_csv("nifty_options_data.csv", index=False)

In [349]:
# List of instrument tokens for which we want quotes

token_map_sample =  token_map #dict(list(token_map.items())[:5])

tokens = list(token_map_sample.values())

# Fetch quotes (can pass a list of instrument tokens as strings or ints)
quotes = kite.quote(tokens)  # or kite.quote(tokens)

quotes_df = pd.DataFrame([
    {
        "symbol": symbol,
        "token": token,
        **quotes[str(token)]
    }
    for symbol, token in token_map_sample.items()
])

display(quotes_df)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=11389954&i=11390978&i=11385090&i=11391490&i=11384578&i=11392002&i=11384066&i=11393026&i=11383042&i=11393538&i=11382530&i=11394050&i=11382018&i=11394562&i=11395586&i=11395074&i=11381506&i=11380994&i=11380482&i=11379458 HTTP/1.1" 200 None


,symbol,token,instrument_token,timestamp,last_trade_time,last_price,last_quantity,buy_quantity,sell_quantity,volume,average_price,oi,oi_day_high,oi_day_low,net_change,lower_circuit_limit,upper_circuit_limit,ohlc,depth
0,NIFTY2581424600CE,11389954,11389954,2025-08-08 14:45:36,2025-08-08 14:45:35,69.10,75,897600,2831175,171153075,86.35,13580550,13666500,3075975,0,0.05,478.00,"{'open': 150, 'high': 160, 'low': 61.4, 'close...","{'buy': [{'price': 68.9, 'quantity': 2400, 'or..."
1,NIFTY2581424650CE,11390978,11390978,2025-08-08 14:45:35,2025-08-08 14:45:34,55.05,150,3064125,925725,65986650,68.91,4400925,4400925,993375,0,0.05,431.20,"{'open': 128, 'high': 128, 'low': 49.05, 'clos...","{'buy': [{'price': 55, 'quantity': 2475, 'orde..."
2,NIFTY2581424500PE,11385090,11385090,2025-08-08 14:45:35,2025-08-08 14:45:34,153.80,150,2865900,602175,235862625,137.66,6512625,10150050,4626675,0,0.05,531.60,"{'open': 84, 'high': 174, 'low': 81, 'close': ...","{'buy': [{'price': 153.5, 'quantity': 525, 'or..."
3,NIFTY2581424700CE,11391490,11391490,2025-08-08 14:45:35,2025-08-08 14:45:34,43.75,75,3034650,1623225,110810175,54.45,9509850,10633275,2872500,0,0.05,383.50,"{'open': 100, 'high': 104.85, 'low': 39.1, 'cl...","{'buy': [{'price': 43.55, 'quantity': 2925, 'o..."
4,NIFTY2581424450PE,11384578,11384578,2025-08-08 14:45:35,2025-08-08 14:45:35,127.65,450,3504675,585975,193880775,120.04,3850500,4755075,1200975,0,0.05,482.90,"{'open': 69.4, 'high': 146.95, 'low': 66.7, 'c...","{'buy': [{'price': 127.35, 'quantity': 1350, '..."
5,NIFTY2581424750CE,11392002,11392002,2025-08-08 14:45:35,2025-08-08 14:45:35,34.40,75,3100575,569775,51497175,43.41,3777150,4086150,988800,0,0.05,342.30,"{'open': 72, 'high': 77.5, 'low': 31, 'close':...","{'buy': [{'price': 34.4, 'quantity': 75, 'orde..."
6,NIFTY2581424400PE,11384066,11384066,2025-08-08 14:45:36,2025-08-08 14:45:35,105.20,150,4147125,1102575,262561425,100.14,7358325,8028225,3588750,0,0.05,436.25,"{'open': 59.3, 'high': 122, 'low': 53.65, 'clo...","{'buy': [{'price': 104.9, 'quantity': 525, 'or..."
7,NIFTY2581424800CE,11393026,11393026,2025-08-08 14:45:35,2025-08-08 14:45:35,27.30,75,3468825,1963125,99405600,33.83,9213900,9483375,3072675,0,0.05,302.95,"{'open': 54.75, 'high': 63.95, 'low': 24.65, '...","{'buy': [{'price': 27.2, 'quantity': 6450, 'or..."
8,NIFTY2581424350PE,11383042,11383042,2025-08-08 14:45:35,2025-08-08 14:45:34,85.75,75,3393150,478575,112253925,81.76,3094500,3274200,1523250,0,0.05,394.10,"{'open': 37.3, 'high': 101.4, 'low': 37.3, 'cl...","{'buy': [{'price': 85.6, 'quantity': 750, 'ord..."
9,NIFTY2581424850CE,11393538,11393538,2025-08-08 14:45:35,2025-08-08 14:45:32,21.55,600,3300375,419325,39044775,26.68,2769675,2844825,852600,0,0.05,268.30,"{'open': 47.8, 'high': 48.25, 'low': 19.7, 'cl...","{'buy': [{'price': 21.55, 'quantity': 8400, 'o..."


#### Function to fetch NIFTY/SENSEX Options

### NIFTY

In [ ]:
def fetch_index_options_weekly_data(kite, index_symbol, start_date, end_date, expiry_date, interval="day", step=None):
    """
    Fetch index historical data (NIFTY 50 or SENSEX), determine ATM strike,
    generate option trading symbols for a given expiry, map them to instrument tokens,
    fetch their historical data, and get latest quotes.

    Parameters
    ----------
    kite : KiteConnect
        Initialized Kite Connect API client instance.
    index_symbol : str
        "SENSEX" or "NIFTY 50".
    start_date : datetime
        Start datetime for historical data.
    end_date : datetime
        End datetime for options historical data.
    expiry_date : datetime
        Expiry date for options trading symbols.
    interval : str, default "day"
        Candle interval for index historical data.
    step : int, optional
        Strike step size (default: 100 for Sensex, 50 for Nifty).

    Returns
    -------
    index_open : int
    index_ltp : float
    atm_strike : int
    token_map : dict
    all_hist_df : DataFrame
    quotes_df : DataFrame
    """

    # ----------------
    #  INDEX METADATA
    # ----------------
    index_map = {
        "SENSEX": {"token": 265, "exchange": "BSE", "step": 100, "opt_segment": "BFO"},
        "NIFTY 50": {"token": 256265, "exchange": "NSE", "step": 50, "opt_segment": "NFO"}
    }

    if index_symbol.upper() not in [key.upper() for key in index_map.keys()]:
        raise ValueError("Invalid index symbol. Use 'SENSEX' or 'NIFTY 50'")

    # Pick correct config
    for k, v in index_map.items():
        if k.upper() == index_symbol.upper():
            index_cfg = v
            index_symbol_std = k  # Standardized case
            break

    if step is None:
        step = index_cfg["step"]

    # ----------------
    # 1. Fetch Index Historical Data
    # ----------------
    hist = kite.historical_data(index_cfg["token"], start_date, start_date, interval="day")
    df_hist = pd.DataFrame(hist)
    df_hist['date'] = pd.to_datetime(df_hist['date'])
    index_open = int(df_hist['open'][0])

    # LTP of index
    full_symbol = f"{index_cfg['exchange']}:{index_symbol_std}"
    index_ltp = kite.quote(full_symbol)[full_symbol]['last_price']

    print("\n---")
    print(f"Current {index_symbol_std} LTP: {index_ltp}")
    print(f"{index_symbol_std} Open on {start_date.date()}: {index_open}")

    # ----------------
    # 2. ATM Calculation
    # ----------------
    atm_strike = int(round(index_open / step) * step)
    print(f"ATM Strike: {atm_strike}")

    # Strike Range
    strike_range = [atm_strike + i * step for i in range(-10, 11)]

    # ----------------
    # 3. Expiry String & Trading Symbols
    #    Exchange code format is different for NSE vs BSE
    # ----------------
    expiry_str_bse = f"{expiry_date.year % 100}{expiry_date.month:d}{expiry_date.day:02d}"  # e.g., 25814
    expiry_str_nse = f"{expiry_date.year % 100}{expiry_date.month:d}{expiry_date.day:02d}"  # '25814'


    trading_symbols = []
    for strike in strike_range:
        if index_cfg['exchange'] == 'BSE':  # SENSEX
            if strike < atm_strike:
                trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}PE")
            elif strike > atm_strike:
                trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}CE")
        else:  # NSE (NIFTY)
            # Force "NIFTY" instead of NIFTY50
            nse_opts_symbol_prefix = "NIFTY" if "NIFTY" in index_symbol_std.upper() else index_symbol_std.replace(' ', '')
            if strike < atm_strike:
                trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}PE")
            elif strike > atm_strike:
                trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}CE")

    # ----------------
    # 4. Map to Tokens
    # ----------------
    try:
        instruments_df = pd.DataFrame(kite.instruments(index_cfg["opt_segment"]))
        index_ops = instruments_df[
            (instruments_df['exchange'] == index_cfg["opt_segment"]) &
            (instruments_df['segment'] == f"{index_cfg['opt_segment']}-OPT") &
            (instruments_df['tradingsymbol'].isin(trading_symbols)) &
            (instruments_df['expiry'] == expiry_date.date())
        ]
        
        token_map = {
            row['tradingsymbol']: row['instrument_token']
            for _, row in index_ops.iterrows()
        }
        print("\nMapped Options Tokens:")
        print(token_map)
    except Exception as e:
        print("\nCould not fetch instruments or no options available.")
        print(str(e))
        token_map = {}

    # ----------------
    # 5. Historical Data for Options
    # ----------------
    market_close_time = time(15, 30)  # 3:30 PM IST
    if start_date.date() == end_date.date():
        if end_date.time() > market_close_time:
            end_date = datetime.combine(end_date.date(), market_close_time)

    all_hist_df = pd.DataFrame()
    for symbol, token in {s: token_map.get(s) for s in trading_symbols}.items():
        if token:
            try:
                hist = kite.historical_data(token, start_date, end_date, interval=interval)
                df_hist = pd.DataFrame(hist)
                df_hist["symbol"] = symbol
                df_hist["token"] = token
                all_hist_df = pd.concat([all_hist_df, df_hist], ignore_index=True)
            except Exception as e:
                print(f"Error fetching data for {symbol}: {e}")
        else:
            print(f"Token not found for {symbol}")

    # For intraday, filter out candles after market close (3:30 PM IST)
        all_hist_df['date'] = pd.to_datetime(all_hist_df['date'])
        market_close_time = time(15, 30)
        all_hist_df = all_hist_df[all_hist_df['date'].dt.time <= market_close_time]

    # ----------------
    # 6. Latest Quotes (Sample)
    # ----------------
    token_map_sample = dict(list(token_map.items())[:5])
    tokens = list(token_map_sample.values())
    if tokens:
        quotes = kite.quote(tokens)
        quotes_df = pd.DataFrame([
            {"symbol": sym, "token": tok, **quotes[str(tok)]}
            for sym, tok in token_map_sample.items()
        ])
    else:
        quotes_df = pd.DataFrame()

    # ----------------
    # 7. Summary
    # ----------------
    print(f"\nTotal option trading symbols generated: {len(trading_symbols)}")
    print(f"Total mapped tokens found: {len(token_map)}")
    print(f"Historical rows fetched for options: {len(all_hist_df)}")
    print(f"Sample quotes fetched: {len(quotes_df)}")

    return index_open, index_ltp, atm_strike, token_map, all_hist_df, quotes_df


In [37]:
def fetch_index_options_weekly_data(kite, index_symbol, start_date, end_date, expiry_date, interval="day", step=None):
    """
    Fetch index historical data (NIFTY 50 or SENSEX), determine ATM strike,
    generate option trading symbols (CE + PE for all strikes, including ATM),
    map them to instrument tokens, fetch their historical data, and get latest quotes.
    """
    import pandas as pd
    from datetime import datetime, time

    # ----------------
    # INDEX METADATA
    # ----------------
    index_map = {
        "SENSEX": {"token": 265, "exchange": "BSE", "step": 100, "opt_segment": "BFO"},
        "NIFTY 50": {"token": 256265, "exchange": "NSE", "step": 50, "opt_segment": "NFO"}
    }

    if index_symbol.upper() not in [key.upper() for key in index_map.keys()]:
        raise ValueError("Invalid index symbol. Use 'SENSEX' or 'NIFTY 50'")

    # Pick correct config
    for k, v in index_map.items():
        if k.upper() == index_symbol.upper():
            index_cfg = v
            index_symbol_std = k
            break

    if step is None:
        step = index_cfg["step"]

    # ----------------
    # 1. Fetch Index Historical Data
    # ----------------
    hist = kite.historical_data(index_cfg["token"], start_date, start_date, interval="day")
    df_hist = pd.DataFrame(hist)
    df_hist['date'] = pd.to_datetime(df_hist['date'])
    index_open = int(df_hist['open'][0])

    # LTP of index
    full_symbol = f"{index_cfg['exchange']}:{index_symbol_std}"
    index_ltp = kite.quote(full_symbol)[full_symbol]['last_price']

    print("\n---")
    print(f"Current {index_symbol_std} LTP: {index_ltp}")
    print(f"{index_symbol_std} Open on {start_date.date()}: {index_open}")

    # ----------------
    # 2. ATM Calculation
    # ----------------
    atm_strike = int(round(index_open / step) * step)
    print(f"ATM Strike: {atm_strike}")

    # Strike Range
    strike_range = [atm_strike + i * step for i in range(-50, 51)]

    # ----------------
    # 3. Expiry String & Trading Symbols
    # ----------------
    expiry_str_bse = f"{expiry_date.year % 100}{'O'}{expiry_date.day:02d}"  # e.g., 25814
    expiry_str_nse = f"{expiry_date.year % 100}{'O'}{expiry_date.day:02d}"  # '25814'


    trading_symbols = []
    for strike in strike_range:
        if index_cfg['exchange'] == 'BSE':  # SENSEX
            trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}CE")
            trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}PE")
        else:  # NSE (NIFTY)
            nse_opts_symbol_prefix = "NIFTY" if "NIFTY" in index_symbol_std.upper() else index_symbol_std.replace(' ', '')
            trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}CE")
            trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}PE")

    # ----------------
    # 4. Map to Tokens
    # ----------------
    try:
        instruments_df = pd.DataFrame(kite.instruments(index_cfg["opt_segment"]))
        index_ops = instruments_df[
            (instruments_df['exchange'] == index_cfg["opt_segment"]) &
            (instruments_df['segment'] == f"{index_cfg['opt_segment']}-OPT") &
            (instruments_df['tradingsymbol'].isin(trading_symbols)) &
            (instruments_df['expiry'] == expiry_date.date())
        ]
        
        token_map = {
            row['tradingsymbol']: row['instrument_token']
            for _, row in index_ops.iterrows()
        }
        print("\nMapped Options Tokens:")
        print(token_map)
    except Exception as e:
        print("\nCould not fetch instruments or no options available.")
        print(str(e))
        token_map = {}

    # ----------------
    # 5. Historical Data for Options
    # ----------------
    market_close_time = time(15, 30)
    if start_date.date() == end_date.date():
        if end_date.time() > market_close_time:
            end_date = datetime.combine(end_date.date(), market_close_time)

    all_hist_df = pd.DataFrame()
    for symbol, token in {s: token_map.get(s) for s in trading_symbols}.items():
        if token:
            try:
                hist = kite.historical_data(token, start_date, end_date, interval=interval)
                df_hist = pd.DataFrame(hist)
                df_hist["symbol"] = symbol
                df_hist["token"] = token
                all_hist_df = pd.concat([all_hist_df, df_hist], ignore_index=True)
            except Exception as e:
                print(f"Error fetching data for {symbol}: {e}")
        else:
            print(f"Token not found for {symbol}")

    # Filter candles after market close
    if not all_hist_df.empty:
        all_hist_df['date'] = pd.to_datetime(all_hist_df['date'])
        all_hist_df = all_hist_df[all_hist_df['date'].dt.time <= market_close_time]

    # ----------------
    # 6. Latest Quotes (sample subset)
    # ----------------
    token_map_sample = dict(list(token_map.items())[:5])
    tokens = list(token_map_sample.values())
    if tokens:
        quotes = kite.quote(tokens)
        quotes_df = pd.DataFrame([
            {"symbol": sym, "token": tok, **quotes[str(tok)]}
            for sym, tok in token_map_sample.items()
        ])
    else:
        quotes_df = pd.DataFrame()


    # ----------------
    # 7. Add Index Value (NIFTY/SENSEX) to all option candles
    # ----------------
    try:
        # Fetch intraday index historical data matching 'interval'
        index_hist = kite.historical_data(
            index_cfg["token"],
            start_date,
            end_date,
            interval=interval
        )
        df_index = pd.DataFrame(index_hist)
        df_index['date'] = pd.to_datetime(df_index['date'])
        df_index.rename(columns={'close': f'index_close'}, inplace=True)

        if not all_hist_df.empty:
            # Merge option candles with index value at same timestamp
            all_hist_df = pd.merge(
                all_hist_df,
                df_index[['date', f'index_close']],
                on="date",
                how="left"
            )
            print(f"\nAdded {index_symbol_std} values as a column to options dataframe.")
        else:
            print("\nOptions dataframe empty, skipping index merge.")

    except Exception as e:
        print(f"\nError adding {index_symbol_std} values to options dataframe: {e}")


    def parse_symbol(symbol):
        # Example symbol: NIFTY25AUG24800PE or NIFTY2581424800PE
        match = re.match(r'([A-Z]+)(\d{2}[A-Z]{3}|\d{5})(\d+)(CE|PE)', symbol)
        if match:
            index = match.group(1)
            expiry_raw = match.group(2)
            strike = match.group(3)
            option_type = match.group(4)
            # Format expiry month
            if len(expiry_raw) == 5:  # e.g., 25814
            #     year = "20" + expiry_raw[:2]
            #     month_num = int(expiry_raw[2])
            #     day = expiry_raw[3:]
            #     month = pd.to_datetime(f"{year}-{month_num}-01").strftime("%b").upper()
            #     expiry_month = f"{expiry_raw[:2]}-{month}-{day}"
            # else:  # e.g., 25AUG
                expiry_month = expiry_raw
            return pd.Series([index, expiry_month, strike, option_type])
        else:
            return pd.Series([None, None, None, None])

    all_hist_df[['index', 'expiry_month', 'strike', 'option_type']] = all_hist_df['symbol'].apply(parse_symbol)


    def add_datetime_and_target_columns(df, expiry_date):
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
        df['date_only'] = df['date'].dt.date
        df['time'] = df['date'].dt.strftime('%H:%M')
        df['AM_PM'] = df['date'].dt.strftime('%p')
        df['day'] = df['date'].dt.strftime('%A')
        # Assign target_date for each symbol
        def get_target(row):
            delta = (expiry_date.date() - row['date_only']).days
            if delta == 0:
                return 'T'
            elif delta > 0:
                return f'T-{delta}'
            else:
                return f'T+{abs(delta)}'
        df['target_date'] = df.apply(get_target, axis=1)
        return df

    # Example usage:
    all_hist_df = add_datetime_and_target_columns(all_hist_df, expiry_date)
    all_hist_df.sort_values(by=['symbol', 'date'], inplace=True)

    def add_index_open_and_atm_strike(all_hist_df, index_open, atm_strike, start_date, end_date):
        # Add columns for index_open and atm_strike to all rows
        all_hist_df['index_open'] = index_open
        all_hist_df['atm_strike'] = atm_strike
        all_hist_df['start_date'] = start_date
        all_hist_df['end_date'] = end_date
        return all_hist_df

    # Usage example:
    all_hist_df = add_index_open_and_atm_strike(all_hist_df, index_open, atm_strike, start_date, end_date)
    all_hist_df['date'] = pd.to_datetime(all_hist_df['date']).dt.tz_localize(None)

    # ----------------
    # 8. Summary
    # ----------------
    print(f"\nTotal option trading symbols generated: {len(trading_symbols)}")
    print(f"Total mapped tokens found: {len(token_map)}")
    print(f"Historical rows fetched for options: {len(all_hist_df)}")
    print(f"Sample quotes fetched: {len(quotes_df)}")

    return index_open, index_ltp, atm_strike, token_map, all_hist_df, quotes_df


In [18]:
def fetch_index_options_monthly_data(kite, index_symbol, start_date, end_date, expiry_date, interval="day", step=None):
    """
    Fetch index historical data (NIFTY 50 or SENSEX), determine ATM strike,
    generate option trading symbols (CE + PE for all strikes, including ATM),
    map them to instrument tokens, fetch their historical data, and get latest quotes.
    """
    import pandas as pd
    from datetime import datetime, time

    # ----------------
    # INDEX METADATA
    # ----------------
    index_map = {
        "SENSEX": {"token": 265, "exchange": "BSE", "step": 100, "opt_segment": "BFO"},
        "NIFTY 50": {"token": 256265, "exchange": "NSE", "step": 50, "opt_segment": "NFO"}
    }

    if index_symbol.upper() not in [key.upper() for key in index_map.keys()]:
        raise ValueError("Invalid index symbol. Use 'SENSEX' or 'NIFTY 50'")

    # Pick correct config
    for k, v in index_map.items():
        if k.upper() == index_symbol.upper():
            index_cfg = v
            index_symbol_std = k
            break

    if step is None:
        step = index_cfg["step"]

    # ----------------
    # 1. Fetch Index Historical Data
    # ----------------
    hist = kite.historical_data(index_cfg["token"], start_date, start_date, interval="day")
    df_hist = pd.DataFrame(hist)
    df_hist['date'] = pd.to_datetime(df_hist['date'])
    index_open = int(df_hist['open'][0])

    # LTP of index
    full_symbol = f"{index_cfg['exchange']}:{index_symbol_std}"
    index_ltp = kite.quote(full_symbol)[full_symbol]['last_price']

    print("\n---")
    print(f"Current {index_symbol_std} LTP: {index_ltp}")
    print(f"{index_symbol_std} Open on {start_date.date()}: {index_open}")

    # ----------------
    # 2. ATM Calculation
    # ----------------
    atm_strike = int(round(index_open / step) * step)
    print(f"ATM Strike: {atm_strike}")

    # Strike Range
    strike_range = [atm_strike + i * step for i in range(-50, 51)]

    # ----------------
    # 3. Expiry String & Trading Symbols
    # ----------------
    expiry_str_bse = f"{expiry_date.year % 100}{expiry_date.strftime('%b').upper()}"  # e.g., 25AUG
    expiry_str_nse = f"{expiry_date.year % 100}{expiry_date.strftime('%b').upper()}"  # e.g., 25AUG

    trading_symbols = []
    for strike in strike_range:
        if index_cfg['exchange'] == 'BSE':  # SENSEX
            trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}CE")
            trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}PE")
        else:  # NSE (NIFTY)
            nse_opts_symbol_prefix = "NIFTY" if "NIFTY" in index_symbol_std.upper() else index_symbol_std.replace(' ', '')
            trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}CE")
            trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}PE")

    # ----------------
    # 4. Map to Tokens
    # ----------------
    try:
        instruments_df = pd.DataFrame(kite.instruments(index_cfg["opt_segment"]))
        index_ops = instruments_df[
            (instruments_df['exchange'] == index_cfg["opt_segment"]) &
            (instruments_df['segment'] == f"{index_cfg['opt_segment']}-OPT") &
            (instruments_df['tradingsymbol'].isin(trading_symbols)) &
            (instruments_df['expiry'] == expiry_date.date())
        ]
        
        token_map = {
            row['tradingsymbol']: row['instrument_token']
            for _, row in index_ops.iterrows()
        }
        print("\nMapped Options Tokens:")
        print(token_map)
    except Exception as e:
        print("\nCould not fetch instruments or no options available.")
        print(str(e))
        token_map = {}

    # ----------------
    # 5. Historical Data for Options
    # ----------------
    market_close_time = time(15, 30)
    if start_date.date() == end_date.date():
        if end_date.time() > market_close_time:
            end_date = datetime.combine(end_date.date(), market_close_time)

    all_hist_df = pd.DataFrame()
    for symbol, token in {s: token_map.get(s) for s in trading_symbols}.items():
        if token:
            try:
                hist = kite.historical_data(token, start_date, end_date, interval=interval)
                df_hist = pd.DataFrame(hist)
                df_hist["symbol"] = symbol
                df_hist["token"] = token
                all_hist_df = pd.concat([all_hist_df, df_hist], ignore_index=True)
            except Exception as e:
                print(f"Error fetching data for {symbol}: {e}")
        else:
            print(f"Token not found for {symbol}")

    # Filter candles after market close
    if not all_hist_df.empty:
        all_hist_df['date'] = pd.to_datetime(all_hist_df['date'])
        all_hist_df = all_hist_df[all_hist_df['date'].dt.time <= market_close_time]

    # ----------------
    # 6. Latest Quotes (sample subset)
    # ----------------
    token_map_sample = dict(list(token_map.items())[:5])
    tokens = list(token_map_sample.values())
    if tokens:
        quotes = kite.quote(tokens)
        quotes_df = pd.DataFrame([
            {"symbol": sym, "token": tok, **quotes[str(tok)]}
            for sym, tok in token_map_sample.items()
        ])
    else:
        quotes_df = pd.DataFrame()


    # ----------------
    # 7. Add Index Value (NIFTY/SENSEX) to all option candles
    # ----------------
    try:
        # Fetch intraday index historical data matching 'interval'
        index_hist = kite.historical_data(
            index_cfg["token"],
            start_date,
            end_date,
            interval=interval
        )
        df_index = pd.DataFrame(index_hist)
        df_index['date'] = pd.to_datetime(df_index['date'])
        df_index.rename(columns={'close': f'index_close'}, inplace=True)

        if not all_hist_df.empty:
            # Merge option candles with index value at same timestamp
            all_hist_df = pd.merge(
                all_hist_df,
                df_index[['date', f'index_close']],
                on="date",
                how="left"
            )
            print(f"\nAdded {index_symbol_std} values as a column to options dataframe.")
        else:
            print("\nOptions dataframe empty, skipping index merge.")

    except Exception as e:
        print(f"\nError adding {index_symbol_std} values to options dataframe: {e}")


    def parse_symbol(symbol):
        # Example symbol: NIFTY25AUG24800PE or NIFTY2581424800PE
        match = re.match(r'([A-Z]+)(\d{2}[A-Z]{3}|\d{5})(\d+)(CE|PE)', symbol)
        if match:
            index = match.group(1)
            expiry_raw = match.group(2)
            strike = match.group(3)
            option_type = match.group(4)
            # Format expiry month
            if len(expiry_raw) == 5:  # e.g., 25814
            #     year = "20" + expiry_raw[:2]
            #     month_num = int(expiry_raw[2])
            #     day = expiry_raw[3:]
            #     month = pd.to_datetime(f"{year}-{month_num}-01").strftime("%b").upper()
            #     expiry_month = f"{expiry_raw[:2]}-{month}-{day}"
            # else:  # e.g., 25AUG
                expiry_month = expiry_raw
            return pd.Series([index, expiry_month, strike, option_type])
        else:
            return pd.Series([None, None, None, None])

    all_hist_df[['index', 'expiry_month', 'strike', 'option_type']] = all_hist_df['symbol'].apply(parse_symbol)


    def add_datetime_and_target_columns(df, expiry_date):
        df = df.copy()
        df['date'] = pd.to_datetime(df['date'])
        df['date_only'] = df['date'].dt.date
        df['time'] = df['date'].dt.strftime('%H:%M')
        df['AM_PM'] = df['date'].dt.strftime('%p')
        df['day'] = df['date'].dt.strftime('%A')
        # Assign target_date for each symbol
        def get_target(row):
            delta = (expiry_date.date() - row['date_only']).days
            if delta == 0:
                return 'T'
            elif delta > 0:
                return f'T-{delta}'
            else:
                return f'T+{abs(delta)}'
        df['target_date'] = df.apply(get_target, axis=1)
        return df

    # Example usage:
    all_hist_df = add_datetime_and_target_columns(all_hist_df, expiry_date)
    all_hist_df.sort_values(by=['symbol', 'date'], inplace=True)

    def add_index_open_and_atm_strike(all_hist_df, index_open, atm_strike, start_date, end_date):
        # Add columns for index_open and atm_strike to all rows
        all_hist_df['index_open'] = index_open
        all_hist_df['atm_strike'] = atm_strike
        all_hist_df['start_date'] = start_date
        all_hist_df['end_date'] = end_date
        return all_hist_df

    # Usage example:
    all_hist_df = add_index_open_and_atm_strike(all_hist_df, index_open, atm_strike, start_date, end_date)
    all_hist_df['date'] = pd.to_datetime(all_hist_df['date']).dt.tz_localize(None)


    # ----------------
    # 8. Summary
    # ----------------
    print(f"\nTotal option trading symbols generated: {len(trading_symbols)}")
    print(f"Total mapped tokens found: {len(token_map)}")
    print(f"Historical rows fetched for options: {len(all_hist_df)}")
    print(f"Sample quotes fetched: {len(quotes_df)}")

    return index_open, index_ltp, atm_strike, token_map, all_hist_df, quotes_df


In [32]:
## NIFTY - Weekly - 5min

start = datetime(2025, 9, 23, 9, 15)
end = datetime(2025, 10, 7, 15, 45)
index = "NIFTY 50"
expiry_type = "weekly"
nifty_expiry = datetime(2025, 10, 7)
interval = "5minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_weekly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = nifty_expiry.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/day?from=2025-09-23+09%3A15%3A00&to=2025-09-23+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=NSE%3ANIFTY+50 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/NFO HTTP/1.1" 200 443378



---
Current NIFTY 50 LTP: 25108.3
NIFTY 50 Open on 2025-09-23: 25209
ATM Strike: 25200


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/9779458/5minute?from=2025-09-23+09%3A15%3A00&to=2025-10-07+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'NIFTY25O0725100CE': 9829122, 'NIFTY25O0725100PE': 9829378, 'NIFTY25O0725050CE': 9828610, 'NIFTY25O0725050PE': 9828866, 'NIFTY25O0725150CE': 9829634, 'NIFTY25O0725150PE': 9829890, 'NIFTY25O0725000CE': 9827074, 'NIFTY25O0725000PE': 9827586, 'NIFTY25O0725200CE': 9830146, 'NIFTY25O0725200PE': 9830402, 'NIFTY25O0724950CE': 9826562, 'NIFTY25O0724950PE': 9826818, 'NIFTY25O0725250CE': 9830658, 'NIFTY25O0725250PE': 9830914, 'NIFTY25O0724900CE': 9823746, 'NIFTY25O0724900PE': 9825026, 'NIFTY25O0725300CE': 9831170, 'NIFTY25O0725300PE': 9831426, 'NIFTY25O0724850CE': 9822978, 'NIFTY25O0724850PE': 9823234, 'NIFTY25O0725350CE': 9832194, 'NIFTY25O0725350PE': 9832450, 'NIFTY25O0724800CE': 9822466, 'NIFTY25O0724800PE': 9822722, 'NIFTY25O0725400CE': 9832706, 'NIFTY25O0725400PE': 9832962, 'NIFTY25O0724750CE': 9821954, 'NIFTY25O0724750PE': 9822210, 'NIFTY25O0725450CE': 9833218, 'NIFTY25O0725450PE': 9833474, 'NIFTY25O0725500PE': 9833986, 'NIFTY25O0727150CE': 9863170, 'NIFTY25O072715

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/9779714/5minute?from=2025-09-23+09%3A15%3A00&to=2025-10-07+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/9779970/5minute?from=2025-09-23+09%3A15%3A00&to=2025-10-07+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/9780226/5minute?from=2025-09-23+09%3A15%3A00&to=2025-10-07+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/9782018/5minute?from=2025-09-23+0

Token not found for NIFTY25O0727200CE
Token not found for NIFTY25O0727200PE
Token not found for NIFTY25O0727250CE
Token not found for NIFTY25O0727250PE
Token not found for NIFTY25O0727300CE
Token not found for NIFTY25O0727300PE
Token not found for NIFTY25O0727350CE
Token not found for NIFTY25O0727350PE
Token not found for NIFTY25O0727400CE
Token not found for NIFTY25O0727400PE
Token not found for NIFTY25O0727450CE
Token not found for NIFTY25O0727450PE
Token not found for NIFTY25O0727500CE
Token not found for NIFTY25O0727500PE
Token not found for NIFTY25O0727550CE
Token not found for NIFTY25O0727550PE
Token not found for NIFTY25O0727600CE
Token not found for NIFTY25O0727600PE
Token not found for NIFTY25O0727650CE
Token not found for NIFTY25O0727650PE
Token not found for NIFTY25O0727700CE
Token not found for NIFTY25O0727700PE


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/5minute?from=2025-09-23+09%3A15%3A00&to=2025-10-07+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None



Added NIFTY 50 values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 174
Historical rows fetched for options: 98764
Sample quotes fetched: 5


In [ ]:
nf_option_hist_df

In [214]:
## NIFTY - Weekly - 15min

start = datetime(2025, 8, 19, 9, 15)
end = datetime(2025, 9, 2, 15, 45)
index = "NIFTY 50"
expiry_type = "weekly"
nifty_expiry = datetime(2025, 9, 2)
interval = "15minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_weekly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = nifty_expiry.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/day?from=2025-08-19+09%3A15%3A00&to=2025-08-19+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=NSE%3ANIFTY+50 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/NFO HTTP/1.1" 200 380417



---
Current NIFTY 50 LTP: 24579.6
NIFTY 50 Open on 2025-08-19: 24891
ATM Strike: 24900


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/12495106/15minute?from=2025-08-19+09%3A15%3A00&to=2025-09-02+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'NIFTY2590224650CE': 12527874, 'NIFTY2590224650PE': 12528130, 'NIFTY2590224600CE': 12527362, 'NIFTY2590224600PE': 12527618, 'NIFTY2590224700CE': 12528386, 'NIFTY2590224700PE': 12528642, 'NIFTY2590224550CE': 12526850, 'NIFTY2590224550PE': 12527106, 'NIFTY2590224750CE': 12528898, 'NIFTY2590224750PE': 12529666, 'NIFTY2590224500CE': 12525826, 'NIFTY2590224500PE': 12526082, 'NIFTY2590224800CE': 12529922, 'NIFTY2590224800PE': 12530178, 'NIFTY2590224450CE': 12525314, 'NIFTY2590224450PE': 12525570, 'NIFTY2590224850CE': 12530434, 'NIFTY2590224850PE': 12530690, 'NIFTY2590224400CE': 12524802, 'NIFTY2590224400PE': 12525058, 'NIFTY2590224900CE': 12530946, 'NIFTY2590224900PE': 12531202, 'NIFTY2590224350CE': 12523266, 'NIFTY2590224350PE': 12524546, 'NIFTY2590224950CE': 12531458, 'NIFTY2590224950PE': 12531714, 'NIFTY2590224300CE': 12522242, 'NIFTY2590224300PE': 12523010, 'NIFTY2590225000CE': 12531970, 'NIFTY2590225000PE': 12532226, 'NIFTY2590225050PE': 12532738, 'NIFTY25902268

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/12496386/15minute?from=2025-08-19+09%3A15%3A00&to=2025-09-02+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/12496642/15minute?from=2025-08-19+09%3A15%3A00&to=2025-09-02+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/12497410/15minute?from=2025-08-19+09%3A15%3A00&to=2025-09-02+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/12497666/15minute?from=2

Token not found for NIFTY2590226900CE
Token not found for NIFTY2590226900PE
Token not found for NIFTY2590226950CE
Token not found for NIFTY2590226950PE
Token not found for NIFTY2590227000CE
Token not found for NIFTY2590227000PE
Token not found for NIFTY2590227050CE
Token not found for NIFTY2590227050PE
Token not found for NIFTY2590227100CE
Token not found for NIFTY2590227100PE
Token not found for NIFTY2590227150CE
Token not found for NIFTY2590227150PE
Token not found for NIFTY2590227200CE
Token not found for NIFTY2590227200PE
Token not found for NIFTY2590227250CE
Token not found for NIFTY2590227250PE
Token not found for NIFTY2590227300CE
Token not found for NIFTY2590227300PE
Token not found for NIFTY2590227350CE
Token not found for NIFTY2590227350PE
Token not found for NIFTY2590227400CE
Token not found for NIFTY2590227400PE


DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=12527874&i=12528130&i=12527362&i=12527618&i=12528386 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/15minute?from=2025-08-19+09%3A15%3A00&to=2025-09-02+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None



Added NIFTY 50 values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 172
Historical rows fetched for options: 32062
Sample quotes fetched: 5


In [19]:
## NIFTY - Monthly - 5min

start = datetime(2025, 9, 16, 9, 15)
end = datetime(2025, 9, 30, 15, 45)
index = "NIFTY 50"
expiry_type = "monthly"
expiry_date = datetime(2025, 9, 30)
interval = "5minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_monthly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = expiry_date.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/day?from=2025-09-16+09%3A15%3A00&to=2025-09-16+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=NSE%3ANIFTY+50 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/NFO HTTP/1.1" 200 451500



---
Current NIFTY 50 LTP: 24611.1
NIFTY 50 Open on 2025-09-16: 25073
ATM Strike: 25050


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'NIFTY25SEP24650CE': 15476482, 'NIFTY25SEP24650PE': 15476738, 'NIFTY25SEP24600CE': 15475970, 'NIFTY25SEP24600PE': 15476226, 'NIFTY25SEP24700CE': 15476994, 'NIFTY25SEP24700PE': 15477762, 'NIFTY25SEP24550CE': 15475458, 'NIFTY25SEP24550PE': 15475714, 'NIFTY25SEP24750CE': 15478018, 'NIFTY25SEP24750PE': 15478786, 'NIFTY25SEP24500CE': 15473410, 'NIFTY25SEP24500PE': 15475202, 'NIFTY25SEP24800CE': 15479042, 'NIFTY25SEP24800PE': 15479298, 'NIFTY25SEP24450CE': 15472898, 'NIFTY25SEP24450PE': 15473154, 'NIFTY25SEP24850CE': 15479554, 'NIFTY25SEP24850PE': 15479810, 'NIFTY25SEP24400CE': 15469314, 'NIFTY25SEP24400PE': 15472642, 'NIFTY25SEP24900CE': 15480066, 'NIFTY25SEP24900PE': 15480322, 'NIFTY25SEP24350CE': 15462914, 'NIFTY25SEP24350PE': 15463938, 'NIFTY25SEP24950CE': 15481346, 'NIFTY25SEP24950PE': 15481602, 'NIFTY25SEP24300CE': 15461890, 'NIFTY25SEP24300PE': 15462658, 'NIFTY25SEP25000CE': 16561666, 'NIFTY25SEP25000PE': 16561922, 'NIFTY25SEP25050PE': 15483394, 'NIFTY25SEP274

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437058/5minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437314/5minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437570/5minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 46
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437826/5minute?from=2025-09-16

Token not found for NIFTY25SEP27450CE
Token not found for NIFTY25SEP27450PE
Token not found for NIFTY25SEP27500CE
Token not found for NIFTY25SEP27500PE
Token not found for NIFTY25SEP27550CE
Token not found for NIFTY25SEP27550PE


DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/5minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None



Added NIFTY 50 values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 194
Historical rows fetched for options: 131141
Sample quotes fetched: 5


In [20]:
## NIFTY - Monthly - 15min

start = datetime(2025, 9, 16, 9, 15)
end = datetime(2025, 9, 30, 15, 45)
index = "NIFTY 50"
expiry_type = "monthly"
expiry_date = datetime(2025, 9, 30)
interval = "15minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_monthly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = expiry_date.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/day?from=2025-09-16+09%3A15%3A00&to=2025-09-16+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=NSE%3ANIFTY+50 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/NFO HTTP/1.1" 200 451500



---
Current NIFTY 50 LTP: 24611.1
NIFTY 50 Open on 2025-09-16: 25073
ATM Strike: 25050


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437058/15minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'NIFTY25SEP24650CE': 15476482, 'NIFTY25SEP24650PE': 15476738, 'NIFTY25SEP24600CE': 15475970, 'NIFTY25SEP24600PE': 15476226, 'NIFTY25SEP24700CE': 15476994, 'NIFTY25SEP24700PE': 15477762, 'NIFTY25SEP24550CE': 15475458, 'NIFTY25SEP24550PE': 15475714, 'NIFTY25SEP24750CE': 15478018, 'NIFTY25SEP24750PE': 15478786, 'NIFTY25SEP24500CE': 15473410, 'NIFTY25SEP24500PE': 15475202, 'NIFTY25SEP24800CE': 15479042, 'NIFTY25SEP24800PE': 15479298, 'NIFTY25SEP24450CE': 15472898, 'NIFTY25SEP24450PE': 15473154, 'NIFTY25SEP24850CE': 15479554, 'NIFTY25SEP24850PE': 15479810, 'NIFTY25SEP24400CE': 15469314, 'NIFTY25SEP24400PE': 15472642, 'NIFTY25SEP24900CE': 15480066, 'NIFTY25SEP24900PE': 15480322, 'NIFTY25SEP24350CE': 15462914, 'NIFTY25SEP24350PE': 15463938, 'NIFTY25SEP24950CE': 15481346, 'NIFTY25SEP24950PE': 15481602, 'NIFTY25SEP24300CE': 15461890, 'NIFTY25SEP24300PE': 15462658, 'NIFTY25SEP25000CE': 16561666, 'NIFTY25SEP25000PE': 16561922, 'NIFTY25SEP25050PE': 15483394, 'NIFTY25SEP274

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437314/15minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437570/15minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 46
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15437826/15minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/15438082/15minute?from=202

Token not found for NIFTY25SEP27450CE
Token not found for NIFTY25SEP27450PE
Token not found for NIFTY25SEP27500CE
Token not found for NIFTY25SEP27500PE
Token not found for NIFTY25SEP27550CE
Token not found for NIFTY25SEP27550PE


DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/15minute?from=2025-09-16+09%3A15%3A00&to=2025-09-30+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None



Added NIFTY 50 values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 194
Historical rows fetched for options: 44001
Sample quotes fetched: 5


In [ ]:
## NIFTY - Weekly - 5min

start = datetime(2025, 8, 22, 9, 15)
end = datetime(2025, 8, 28, 15, 45)
index = "NIFTY 50"
expiry_type = "weekly"
nifty_expiry = datetime(2025, 8, 28)
interval = "5minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_monthly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = expiry_date.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

In [ ]:
## NIFTY - Weekly - 15min

start = datetime(2025, 8, 22, 9, 15)
end = datetime(2025, 8, 28, 15, 45)
index = "NIFTY 50"
expiry_type = "weekly"
nifty_expiry = datetime(2025, 8, 28)
interval = "15minute"

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_monthly_data(kite, index, start, end, nifty_expiry,interval)

expiry1_date = expiry_date.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    nf_option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    nf_quotes_df.to_excel(writer, sheet_name='quotes', index=False)

In [ ]:
## NIFTY - Weekly Expiry

start = datetime(2025, 8, 22, 9, 15)
end = datetime(2025, 8, 28, 15, 30)

nifty_expiry = datetime(2025, 8, 28)

# For NIFTY
nifty_open, nifty_ltp, nf_atm_strike, nf_token_map, nf_option_hist_df, nf_quotes_df  = fetch_index_options_weekly_data(kite, "NIFTY 50", start, end, nifty_expiry,interval="day")

nf_option_hist_df['date'] = pd.to_datetime(nf_option_hist_df['date']).dt.tz_localize(None)
nf_option_hist_df.to_excel('nifty_options_data_15mM258.xlsx', index=False)

### SENSEX

In [39]:
## SENSEX - Weekly - 5min

start = datetime(2025, 9, 25, 9, 15)
end = datetime(2025, 10, 9, 15, 45)
index = "SENSEX"
expiry_type = "weekly"
expiry = datetime(2025, 10, 9)
interval = "15minute"

# For SENSEX
open, ltp, atm_strike, token_map, option_hist_df, quotes_df  = fetch_index_options_weekly_data(kite, index, start, end, expiry,interval)

expiry1_date = expiry.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/day?from=2025-09-25+09%3A15%3A00&to=2025-09-25+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=BSE%3ASENSEX HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/BFO HTTP/1.1" 200 68393



---
Current SENSEX LTP: 82172.1
SENSEX Open on 2025-09-25: 81574
ATM Strike: 81600


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/223558405/15minute?from=2025-09-25+09%3A15%3A00&to=2025-10-09+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 46
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'SENSEX25O0981800CE': 223499269, 'SENSEX25O0981800PE': 223544837, 'SENSEX25O0981700PE': 223558661, 'SENSEX25O0981700CE': 223563269, 'SENSEX25O0981900PE': 223485957, 'SENSEX25O0981900CE': 223527429, 'SENSEX25O0981600CE': 223481861, 'SENSEX25O0981600PE': 223525637, 'SENSEX25O0982000PE': 223495429, 'SENSEX25O0982000CE': 223500037, 'SENSEX25O0981500PE': 223497221, 'SENSEX25O0981500CE': 223540485, 'SENSEX25O0982100PE': 223486981, 'SENSEX25O0982100CE': 223528709, 'SENSEX25O0981400PE': 223508997, 'SENSEX25O0981400CE': 223553797, 'SENSEX25O0982200PE': 223514629, 'SENSEX25O0982200CE': 223561477, 'SENSEX25O0981300PE': 223480069, 'SENSEX25O0981300CE': 223523077, 'SENSEX25O0982300PE': 223503365, 'SENSEX25O0982300CE': 223547909, 'SENSEX25O0981200PE': 223491589, 'SENSEX25O0981200CE': 223494149, 'SENSEX25O0982400CE': 223488773, 'SENSEX25O0982400PE': 223531525, 'SENSEX25O0981100CE': 223505669, 'SENSEX25O0981100PE': 223551493, 'SENSEX25O0982500CE': 223475717, 'SENSEX25O0982500P

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/223511557/15minute?from=2025-09-25+09%3A15%3A00&to=2025-10-09+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/223544581/15minute?from=2025-09-25+09%3A15%3A00&to=2025-10-09+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 46
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/223539461/15minute?from=2025-09-25+09%3A15%3A00&to=2025-10-09+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/223485701/15minute?from


Added SENSEX values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 202
Historical rows fetched for options: 28863
Sample quotes fetched: 5


In [257]:
## SENSEX - Weekly - 15min

start = datetime(2025, 9, 4, 9, 15)
end = datetime(2025, 9, 18, 15, 45)
index = "SENSEX"
expiry_type = "weekly"
expiry = datetime(2025, 9, 18)
interval = "15minute"

# For SENSEX
open, ltp, atm_strike, token_map, option_hist_df, quotes_df  = fetch_index_options_weekly_data(kite, index, start, end, expiry,interval)

expiry1_date = expiry.strftime("%Y%m%d")
with pd.ExcelWriter(f'{index}_{expiry_type}_{interval}_{expiry1_date}.xlsx') as writer:
    option_hist_df.to_excel(writer, sheet_name='option_history', index=False)
    quotes_df.to_excel(writer, sheet_name='quotes', index=False)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/day?from=2025-09-04+09%3A15%3A00&to=2025-09-04+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=BSE%3ASENSEX HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/BFO HTTP/1.1" 200 68053



---
Current SENSEX LTP: 83013.96
SENSEX Open on 2025-09-04: 81456
ATM Strike: 81500


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'SENSEX2591882700CE': 220750085, 'SENSEX2591882700PE': 220780549, 'SENSEX2591882600PE': 220738309, 'SENSEX2591882600CE': 220784133, 'SENSEX2591882800CE': 220717061, 'SENSEX2591882800PE': 220746245, 'SENSEX2591882500CE': 220743173, 'SENSEX2591882500PE': 220773637, 'SENSEX2591882900CE': 220758789, 'SENSEX2591882900PE': 220789253, 'SENSEX2591882400CE': 220776709, 'SENSEX2591882400PE': 220806405, 'SENSEX2591883000CE': 220701445, 'SENSEX2591883000PE': 220731141, 'SENSEX2591882300CE': 220733701, 'SENSEX2591882300PE': 220762373, 'SENSEX2591883100PE': 220698629, 'SENSEX2591883100CE': 220806661, 'SENSEX2591882200CE': 220767749, 'SENSEX2591882200PE': 220797189, 'SENSEX2591883200CE': 220711429, 'SENSEX2591883200PE': 220741381, 'SENSEX2591882100PE': 220752645, 'SENSEX2591882100CE': 220802309, 'SENSEX2591883300CE': 220678405, 'SENSEX2591883300PE': 220706565, 'SENSEX2591882000CE': 220756997, 'SENSEX2591882000PE': 220787461, 'SENSEX2591883400PE': 220675077, 'SENSEX2591883400C

DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/220714757/15minute?from=2025-09-04+09%3A15%3A00&to=2025-09-18+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/220744709/15minute?from=2025-09-04+09%3A15%3A00&to=2025-09-18+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/220755461/15minute?from=2025-09-04+09%3A15%3A00&to=2025-09-18+15%3A45%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/220710405/15minute?fr


Added SENSEX values as a column to options dataframe.

Total option trading symbols generated: 202
Total mapped tokens found: 202
Historical rows fetched for options: 30116
Sample quotes fetched: 5


In [14]:
def fetch_index_options_data_monthly_expiry(kite, index_symbol, start_date, end_date, expiry_date, interval="day", step=None):
    """
    Fetch index historical data (NIFTY 50 or SENSEX), determine ATM strike,
    generate option trading symbols for a given expiry, map them to instrument tokens,
    fetch their historical data, and get latest quotes.

    Parameters
    ----------
    kite : KiteConnect
        Initialized Kite Connect API client instance.
    index_symbol : str
        "SENSEX" or "NIFTY 50".
    start_date : datetime
        Start datetime for historical data.
    end_date : datetime
        End datetime for options historical data.
    expiry_date : datetime
        Expiry date for options trading symbols.
    interval : str, default "day"
        Candle interval for index historical data.
    step : int, optional
        Strike step size (default: 100 for Sensex, 50 for Nifty).

    Returns
    -------
    index_open : int
    index_ltp : float
    atm_strike : int
    token_map : dict
    all_hist_df : DataFrame
    quotes_df : DataFrame
    """

    # ----------------
    #  INDEX METADATA
    # ----------------
    index_map = {
        "SENSEX": {"token": 265, "exchange": "BSE", "step": 100, "opt_segment": "BFO"},
        "NIFTY 50": {"token": 256265, "exchange": "NSE", "step": 50, "opt_segment": "NFO"}
    }

    if index_symbol.upper() not in [key.upper() for key in index_map.keys()]:
        raise ValueError("Invalid index symbol. Use 'SENSEX' or 'NIFTY 50'")

    # Pick correct config
    for k, v in index_map.items():
        if k.upper() == index_symbol.upper():
            index_cfg = v
            index_symbol_std = k  # Standardized case
            break

    if step is None:
        step = index_cfg["step"]

    # ----------------
    # 1. Fetch Index Historical Data
    # ----------------
    hist = kite.historical_data(index_cfg["token"], start_date, start_date, interval="day")
    df_hist = pd.DataFrame(hist)
    df_hist['date'] = pd.to_datetime(df_hist['date'])
    index_open = int(df_hist['open'][0])

    # LTP of index
    full_symbol = f"{index_cfg['exchange']}:{index_symbol_std}"
    index_ltp = kite.quote(full_symbol)[full_symbol]['last_price']

    print("\n---")
    print(f"Current {index_symbol_std} LTP: {index_ltp}")
    print(f"{index_symbol_std} Open on {start_date.date()}: {index_open}")

    # ----------------
    # 2. ATM Calculation
    # ----------------
    atm_strike = int(round(index_open / step) * step)
    print(f"ATM Strike: {atm_strike}")

    # Strike Range
    strike_range = [atm_strike + i * step for i in range(-10, 11)]

    # ----------------
    # 3. Expiry String & Trading Symbols
    #    Exchange code format is different for NSE vs BSE
    # ----------------
    expiry_str_bse = f"{expiry_date.year % 100}{expiry_date.strftime('%b').upper()}"  # e.g., 25AUG
    expiry_str_nse = f"{expiry_date.year % 100}{expiry_date.strftime('%b').upper()}"  # e.g., 25AUG

    trading_symbols = []
    for strike in strike_range:
        if index_cfg['exchange'] == 'BSE':  # SENSEX
            if strike < atm_strike:
                trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}PE")
            elif strike > atm_strike:
                trading_symbols.append(f"{index_symbol_std}{expiry_str_bse}{strike}CE")
        else:  # NSE (NIFTY)
            # Force "NIFTY" instead of NIFTY50
            nse_opts_symbol_prefix = "NIFTY" if "NIFTY" in index_symbol_std.upper() else index_symbol_std.replace(' ', '')
            if strike < atm_strike:
                trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}PE")
            elif strike > atm_strike:
                trading_symbols.append(f"{nse_opts_symbol_prefix}{expiry_str_nse}{strike}CE")

    # ----------------
    # 4. Map to Tokens
    # ----------------
    try:
        instruments_df = pd.DataFrame(kite.instruments(index_cfg["opt_segment"]))
        index_ops = instruments_df[
            (instruments_df['exchange'] == index_cfg["opt_segment"]) &
            (instruments_df['segment'] == f"{index_cfg['opt_segment']}-OPT") &
            (instruments_df['tradingsymbol'].isin(trading_symbols)) &
            (instruments_df['expiry'] == expiry_date.date())
        ]
        
        token_map = {
            row['tradingsymbol']: row['instrument_token']
            for _, row in index_ops.iterrows()
        }
        print("\nMapped Options Tokens:")
        print(token_map)
    except Exception as e:
        print("\nCould not fetch instruments or no options available.")
        print(str(e))
        token_map = {}

    # ----------------
    # 5. Historical Data for Options
    # ----------------
    market_close_time = time(15, 30)  # 3:30 PM IST
    if start_date.date() == end_date.date():
        if end_date.time() > market_close_time:
            end_date = datetime.combine(end_date.date(), market_close_time)

    all_hist_df = pd.DataFrame()
    for symbol, token in {s: token_map.get(s) for s in trading_symbols}.items():
        if token:
            try:
                hist = kite.historical_data(token, start_date, end_date, interval=interval)
                df_hist = pd.DataFrame(hist)
                df_hist["symbol"] = symbol
                df_hist["token"] = token
                all_hist_df = pd.concat([all_hist_df, df_hist], ignore_index=True)
            except Exception as e:
                print(f"Error fetching data for {symbol}: {e}")
        else:
            print(f"Token not found for {symbol}")

    # For intraday, filter out candles after market close (3:30 PM IST)
        all_hist_df['date'] = pd.to_datetime(all_hist_df['date'])
        market_close_time = time(15, 30)
        all_hist_df = all_hist_df[all_hist_df['date'].dt.time <= market_close_time]

    # ----------------
    # 6. Latest Quotes (Sample)
    # ----------------
    token_map_sample = dict(list(token_map.items())[:5])
    tokens = list(token_map_sample.values())
    if tokens:
        quotes = kite.quote(tokens)
        quotes_df = pd.DataFrame([
            {"symbol": sym, "token": tok, **quotes[str(tok)]}
            for sym, tok in token_map_sample.items()
        ])
    else:
        quotes_df = pd.DataFrame()

    # ----------------
    # 7. Summary
    # ----------------
    print(f"\nTotal option trading symbols generated: {len(trading_symbols)}")
    print(f"Total mapped tokens found: {len(token_map)}")
    print(f"Historical rows fetched for options: {len(all_hist_df)}")
    print(f"Sample quotes fetched: {len(quotes_df)}")

    return index_open, index_ltp, atm_strike, token_map, all_hist_df, quotes_df


In [28]:
# SENSEX

start = datetime(2025, 7, 29, 9, 15)
end = datetime(2025, 8, 26, 15, 30)

sensex_expiry = datetime(2025, 8, 26)

# For SENSEX
#sensex_results 
sensex_open, sensex_ltp, sn_atm_strike, sn_token_map, sn_option_hist_df, sn_quotes_df = fetch_index_options_data_monthly_expiry(kite, "SENSEX", start, end, sensex_expiry, interval="15minute")

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/day?from=2025-07-29+09%3A15%3A00&to=2025-07-29+09%3A15%3A00&interval=day&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /quote?i=BSE%3ASENSEX HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/BFO HTTP/1.1" 200 62398



---
Current SENSEX LTP: 80786.54
SENSEX Open on 2025-07-29: 80620
ATM Strike: 80600


DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/211706629/15minute?from=2025-07-29+09%3A15%3A00&to=2025-08-26+15%3A30%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443



Mapped Options Tokens:
{'SENSEX25AUG81600CE': 211607813, 'SENSEX25AUG81500CE': 211671301, 'SENSEX25AUG81400CE': 211737861, 'SENSEX25AUG81300CE': 211656197, 'SENSEX25AUG81200CE': 211721733, 'SENSEX25AUG81100CE': 211637765, 'SENSEX25AUG81000CE': 211703045, 'SENSEX25AUG80900CE': 211816453, 'SENSEX25AUG80800CE': 211732485, 'SENSEX25AUG80700CE': 211796229, 'SENSEX25AUG80500PE': 211569157, 'SENSEX25AUG80400PE': 211636229, 'SENSEX25AUG80300PE': 211824645, 'SENSEX25AUG80200PE': 211619589, 'SENSEX25AUG80100PE': 211803909, 'SENSEX25AUG80000PE': 211603717, 'SENSEX25AUG79900PE': 211659781, 'SENSEX25AUG79800PE': 211573509, 'SENSEX25AUG79700PE': 211641605, 'SENSEX25AUG79600PE': 211706629}


DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/211641605/15minute?from=2025-07-29+09%3A15%3A00&to=2025-08-26+15%3A30%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/211573509/15minute?from=2025-07-29+09%3A15%3A00&to=2025-08-26+15%3A30%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/211659781/15minute?from=2025-07-29+09%3A15%3A00&to=2025-08-26+15%3A30%3A00&interval=15minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/211603717/15minute?fr


Total option trading symbols generated: 20
Total mapped tokens found: 20
Historical rows fetched for options: 9724
Sample quotes fetched: 5


### NIFTY Historical Data

In [64]:
def fetch_historical_data_batchwise(token, symbol, start_dt, end_dt, interval):
    combined_df = pd.DataFrame()
    delta_days = 100

    batch_start = start_dt
    while batch_start < end_dt:
        # Set batch tentative end
        batch_end_candidate = batch_start + timedelta(days=delta_days)
        
        # Adjust batch end
        if batch_end_candidate > end_dt:
            batch_end = end_dt
        else:
            batch_end = datetime.combine(batch_end_candidate.date(), time(15, 30))
            if batch_end > end_dt:
                batch_end = end_dt
        
        # Fetch data from Kite
        hist = kite.historical_data(token, batch_start, batch_end, interval)
        batch_df = pd.DataFrame(hist)

        # Add batch start/end columns
        batch_df["symbol"] = symbol
        #batch_df["token"] = token

        # Combine
        combined_df = pd.concat([combined_df, batch_df], ignore_index=True)

        # Next batch start: 9:15 next day
        next_day = batch_end.date() + timedelta(days=1)
        batch_start = datetime.combine(next_day, time(9, 15))
    
    return combined_df

In [65]:
# Inputs for NIFTY historical data
start = datetime(2024, 8, 29, 9, 15)
end = datetime(2025, 9, 30, 15, 45)
token = 256265
symbol = "NIFTY 50"
interval = "5minute"


# Fetch in batches
df_hist_nifty = fetch_historical_data_batchwise(token, symbol, start, end, interval)
display(df_hist_nifty)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/5minute?from=2024-08-29+09%3A15%3A00&to=2024-12-07+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/5minute?from=2024-12-08+09%3A15%3A00&to=2025-03-18+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/256265/5minute?from=2025-03-19+09%3A15%3A00&to=2025-06-27+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://a

,date,open,high,low,close,volume,symbol
0,2024-08-29 09:15:00+05:30,25035.30,25045.70,25017.50,25029.00,0,NIFTY 50
1,2024-08-29 09:20:00+05:30,25029.60,25051.35,25027.50,25051.35,0,NIFTY 50
2,2024-08-29 09:25:00+05:30,25052.10,25073.65,25049.00,25054.25,0,NIFTY 50
3,2024-08-29 09:30:00+05:30,25054.85,25064.25,25043.85,25046.65,0,NIFTY 50
4,2024-08-29 09:35:00+05:30,25046.45,25063.90,25043.70,25054.30,0,NIFTY 50
...,...,...,...,...,...,...,...
20332,2025-09-30 15:05:00+05:30,24614.20,24617.95,24598.15,24610.65,0,NIFTY 50
20333,2025-09-30 15:10:00+05:30,24608.40,24615.85,24603.95,24614.85,0,NIFTY 50
20334,2025-09-30 15:15:00+05:30,24613.50,24614.75,24603.30,24607.40,0,NIFTY 50
20335,2025-09-30 15:20:00+05:30,24606.95,24613.70,24605.15,24611.85,0,NIFTY 50


In [66]:
df_hist_nifty['date'] = pd.to_datetime(df_hist_nifty['date']).dt.tz_localize(None)

df_hist_nifty

,date,open,high,low,close,volume,symbol
0,2024-08-29 09:15:00,25035.30,25045.70,25017.50,25029.00,0,NIFTY 50
1,2024-08-29 09:20:00,25029.60,25051.35,25027.50,25051.35,0,NIFTY 50
2,2024-08-29 09:25:00,25052.10,25073.65,25049.00,25054.25,0,NIFTY 50
3,2024-08-29 09:30:00,25054.85,25064.25,25043.85,25046.65,0,NIFTY 50
4,2024-08-29 09:35:00,25046.45,25063.90,25043.70,25054.30,0,NIFTY 50
...,...,...,...,...,...,...,...
20332,2025-09-30 15:05:00,24614.20,24617.95,24598.15,24610.65,0,NIFTY 50
20333,2025-09-30 15:10:00,24608.40,24615.85,24603.95,24614.85,0,NIFTY 50
20334,2025-09-30 15:15:00,24613.50,24614.75,24603.30,24607.40,0,NIFTY 50
20335,2025-09-30 15:20:00,24606.95,24613.70,24605.15,24611.85,0,NIFTY 50


In [67]:
df_hist_nifty.to_csv("nifty_historical_data_sep_24-sep_25_5min.csv", index=False)

In [36]:
# Inputs for SENSEX historical data
start = datetime(2023, 1, 1, 9, 15)
end = datetime(2025, 8, 14, 15, 30)
token = 265
symbol = "SENSEX"
interval = "5minute"


# Fetch in batches
df_hist_sensex = fetch_historical_data_batchwise(token, symbol, start, end, interval)
display(df_hist_sensex)

DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/5minute?from=2023-01-01+09%3A15%3A00&to=2023-04-11+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/5minute?from=2023-04-12+09%3A15%3A00&to=2023-07-21+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.trade:443 "GET /instruments/historical/265/5minute?from=2023-07-22+09%3A15%3A00&to=2023-10-30+15%3A30%3A00&interval=5minute&continuous=0&oi=0 HTTP/1.1" 200 None
DEBUG:urllib3.connectionpool:Starting new HTTPS connection (1): api.kite.trade:443
DEBUG:urllib3.connectionpool:https://api.kite.t

,date,open,high,low,close,volume,symbol,token
0,2023-01-02 09:15:00+05:30,60871.24,60963.11,60853.18,60853.18,0,SENSEX,265
1,2023-01-02 09:20:00+05:30,60855.77,60925.84,60789.96,60919.41,0,SENSEX,265
2,2023-01-02 09:25:00+05:30,60916.24,60926.41,60788.91,60809.55,0,SENSEX,265
3,2023-01-02 09:30:00+05:30,60806.23,60900.86,60765.02,60884.40,0,SENSEX,265
4,2023-01-02 09:35:00+05:30,60879.63,60978.47,60866.03,60978.47,0,SENSEX,265
...,...,...,...,...,...,...,...,...
48583,2025-08-14 15:05:00+05:30,80627.51,80651.66,80609.41,80625.36,0,SENSEX,265
48584,2025-08-14 15:10:00+05:30,80624.59,80641.72,80597.08,80597.25,0,SENSEX,265
48585,2025-08-14 15:15:00+05:30,80595.27,80618.40,80574.04,80597.97,0,SENSEX,265
48586,2025-08-14 15:20:00+05:30,80594.78,80618.07,80562.81,80593.07,0,SENSEX,265


In [37]:
# Convert 'date' column to datetime and remove timezone information
df_hist_nifty['date'] = pd.to_datetime(df_hist_nifty['date']).dt.tz_localize(None)
df_hist_sensex['date'] = pd.to_datetime(df_hist_sensex['date']).dt.tz_localize(None)

df_hist_nifty.to_excel("nifty_historical_data_23-25_5min.xlsx", index=False)
df_hist_sensex.to_excel("sensex_historical_data_23-25_5min.xlsx", index=False)

In [ ]:
def feature_engineering_nifty(df):
    df = df.copy()
    # Parse datetime and set as index
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date')
    df.set_index('date', inplace=True)

    # Time-based features
    df['hour'] = df.index.hour
    df['minute'] = df.index.minute

    # Price action features
    df['typical_price'] = (df['high'] + df['low'] + df['close']) / 3
    df['price_range'] = df['high'] - df['low']
    df['return'] = df['close'].pct_change()

    # Lag features
    for lag in range(1, 6):
        df[f'close_lag_{lag}'] = df['close'].shift(lag)
        df[f'return_lag_{lag}'] = df['return'].shift(lag)

    # Rolling volatility
    df['volatility_15'] = df['return'].rolling(window=15).std()

    # Expiry day composite features (NSE expiry is last Thursday monthly)
    df['weekday'] = df.index
    df['month'] = df.index.month
    df['year'] = df.index.year

    # Find last Thursdays
    last_thursdays = []
    for year in df['year'].unique():
        for month in df[df['year'] == year]['month'].unique():
            temp = df[(df['year'] == year) & (df['month'] == month)]
            thursdays = temp[temp['weekday'] == 3].index.floor('D').unique()
            if len(thursdays): last_thursdays.append(thursdays[-1])

    df['expiry_day'] = df.index.floor('D').isin(last_thursdays).astype(int)
    df['day_category'] = 'other_days'
    for expiry in last_thursdays:
        prev = expiry - pd.Timedelta(days=1)
        df.loc[df.index.floor('D') == expiry, 'day_category'] = 'expiry_day'
        df.loc[df.index.floor('D') == prev, 'day_category'] = 'previous_day'

    # Last 2 hours flag (market close usually 15:30 IST)
    df['minutes_since_open'] = (df.index.hour - 9) * 60 + df.index.minute
    market_close_minutes = 15*60 + 30
    df['last_2_hours'] = ((df['minutes_since_open'] >= market_close_minutes - 120) & 
                          (df['minutes_since_open'] <= market_close_minutes)).astype(int)
    return df


In [495]:
# Usage:
df1_hist_nifty = feature_engineering_nifty(df_hist_nifty)


In [489]:
df1_hist_nifty.columns

Index(['open', 'high', 'low', 'close', 'volume', 'symbol', 'token', 'hour',
       'minute', 'typical_price', 'price_range', 'return', 'close_lag_1',
       'return_lag_1', 'close_lag_2', 'return_lag_2', 'close_lag_3',
       'return_lag_3', 'close_lag_4', 'return_lag_4', 'close_lag_5',
       'return_lag_5', 'volatility_15', 'weekday', 'month', 'year',
       'expiry_day', 'day_category', 'minutes_since_open', 'last_2_hours'],
      dtype='object')

In [497]:
df1_hist_nifty[[ 'price_range', 'return', 'close_lag_1','weekday',
       'return_lag_1', 'close_lag_2', 'return_lag_2', 'close_lag_3',
       'return_lag_3', 'close_lag_4', 'return_lag_4', 'close_lag_5',
       'return_lag_5', 'volatility_15', 'weekday', 'month', 'year',
       'expiry_day', 'day_category', 'minutes_since_open', 'last_2_hours']].head()

,price_range,return,close_lag_1,weekday,return_lag_1,close_lag_2,return_lag_2,close_lag_3,return_lag_3,close_lag_4,...,close_lag_5,return_lag_5,volatility_15,weekday,month,year,expiry_day,day_category,minutes_since_open,last_2_hours
date,,,,,,,,,,,,,,,,,,,,,
2024-01-01 09:15:00,52.60,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,1,2024,0,other_days,15,0
2024-01-01 09:30:00,27.85,0.000242,21700.80,0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,1,2024,0,other_days,30,0
2024-01-01 09:45:00,34.35,0.000933,21706.05,0,0.000242,21700.80,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,0,1,2024,0,other_days,45,0
2024-01-01 10:00:00,29.05,-0.000766,21726.30,0,0.000933,21706.05,0.000242,21700.80,NaN,NaN,...,NaN,NaN,NaN,0,1,2024,0,other_days,60,0
2024-01-01 10:15:00,24.30,0.000131,21709.65,0,-0.000766,21726.30,0.000933,21706.05,0.000242,21700.8,...,NaN,NaN,NaN,0,1,2024,0,other_days,75,0
